# Part A

**Ομάδα:**
- Αθανάσιος Νικολέτας [sdi2300140]
- Αχιλλέας Πετρουλάκης [sdi2300171]


## 1.Installing dependances

In [ ]:
%pip install seaborn
%pip install datasets
%pip install matplotlib
%pip install vaderSentiment
%pip install kneed
%pip install contractions
%pip install pyspellchecker
%pip install scikit-learn
%pip install pandas

## 2. Loading the files and extracting data

In [ ]:
import pandas as pd
import os
from pathlib import Path

RAW_DATA = Path('../data/raw')
LYRICS_DIR = Path('../data/processed_lyrics')
PROCESSED_DATA = Path('../data/processed')
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

id_genres = pd.read_csv(RAW_DATA / 'id_genres.csv',sep='\t')
id_information = pd.read_csv(RAW_DATA / 'id_information.csv',sep='\t')
id_tags = pd.read_csv(RAW_DATA / 'id_tags.csv',sep='\t')

chunks = []
for chunk in pd.read_csv(RAW_DATA / 'id_mfcc_stats.tsv',sep='\t',chunksize=5000):
    chunks.append(chunk)
id_mfcc_stats = pd.concat(chunks, ignore_index=True)

print("id_genres:", id_genres.shape)
print("id_information:", id_information.shape)
print("id_tags:", id_tags.shape)
print("id_mfcc_stats:", id_mfcc_stats.shape)

display(id_genres.head(3))
display(id_mfcc_stats.head(3))

In [ ]:
# Get lyrics IDs from filenames
lyrics_ids = {f.stem for f in LYRICS_DIR.iterdir() if f.suffix == '.txt'}
print(f"Lyrics files available: {len(lyrics_ids)}")

# Explode multi-genre, find top-5
genre_exploded = id_genres.assign(
    genre=id_genres['genres'].str.split(',')
).explode('genre')
genre_exploded['genre'] = genre_exploded['genre'].str.strip()

top5_genres = genre_exploded['genre'].value_counts().head(5)
print("\nTop-5 Genres:")
print(top5_genres)
top5_list = top5_genres.index.tolist()

# Filter + intersect
filtered_ids = genre_exploded.loc[genre_exploded['genre'].isin(top5_list), 'id'].unique()
mfcc_ids = set(id_mfcc_stats['id'])
common_ids = set(filtered_ids) & mfcc_ids & lyrics_ids
print(f"\nSongs in all 3 sources: {len(common_ids)}")

# Read lyrics for common IDs only
lyrics_data = []
for song_id in common_ids:
    filepath = LYRICS_DIR / f"{song_id}.txt"
    try:
        text = filepath.read_text(encoding='utf-8').strip()
        lyrics_data.append({'id': song_id, 'lyrics': text})
    except Exception:
        pass

df_lyrics = pd.DataFrame(lyrics_data)

# Build genre map + merge into final df
genre_map = genre_exploded[genre_exploded['genre'].isin(top5_list)].drop_duplicates(subset='id', keep='first')[['id', 'genre']]

df = (
    genre_map[genre_map['id'].isin(common_ids)]
    .merge(df_lyrics, on='id', how='inner')
    .merge(id_mfcc_stats[id_mfcc_stats['id'].isin(common_ids)], on='id', how='inner')
)

print(f"\nFinal dataset: {df.shape[0]} songs, {df.shape[1]} columns")
print(df['genre'].value_counts())
display(df.head())

In [ ]:
import nltk
import numpy as np
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

def preprocess_lyrics(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # keep only letters
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return tokens

# Tokenize all lyrics
df['tokens'] = df['lyrics'].apply(preprocess_lyrics)

print(f"Example tokens: {df['tokens'].iloc[0][:10]}")
print(f"Avg tokens per song: {df['tokens'].apply(len).mean():.0f}")

# Train Word2Vec on lyrics corpus
w2v_model = Word2Vec(
    sentences=df['tokens'].tolist(),
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=20
)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector size: {w2v_model.wv.vector_size}")

# Quick test — similar words to "love"
print(f"\nMost similar to 'love': {w2v_model.wv.most_similar('love', topn=5)}")

# Create document embeddings — average of word vectors per song
def get_doc_embedding(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    return np.zeros(model.wv.vector_size)

text_embeddings = np.array(df['tokens'].apply(lambda t: get_doc_embedding(t, w2v_model)).tolist())

print(f"Text embeddings shape: {text_embeddings.shape}")
# Should be (n_songs, 100)